# Week 20, Day 2: Mutual information: what a response says about a stimulus

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w020/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Mutual information from a joint table

Write `entropy(p)` (as yesterday) and `mutual_info(joint)` using the double-sum form, skipping zero entries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import lgamma

def entropy(p):
    p = np.asarray(p, float); p = p[p > 0]
    return -np.sum(p * np.log2(p))

def mutual_info(joint):
    joint = np.asarray(joint, float)
    ps, pr = joint.sum(axis=1, keepdims=True), joint.sum(axis=0, keepdims=True)
    nz = joint > 0
    return ...

print(mutual_info(np.array([[0.3, 0.1], [0.05, 0.55]])))

In [ ]:
import numpy as np
def test_independent():
    "independent variables have zero mutual information"
    assert abs(mutual_info(np.outer([0.3, 0.7], [0.5, 0.5]))) < 1e-12

def test_identical():
    "a variable shares its full entropy with itself"
    assert abs(mutual_info(np.diag([0.25] * 4)) - 2) < 1e-12

def test_forms():
    "H(R) - H(R | S) gives the same value as the double sum"
    J = np.array([[0.3, 0.1], [0.05, 0.55]])
    pr, ps = J.sum(0), J.sum(1)
    h_r_given_s = sum(ps[i] * entropy(J[i] / ps[i]) for i in range(2))
    assert abs(mutual_info(J) - (entropy(pr) - h_r_given_s)) < 1e-12

test_independent()
test_identical()
test_forms()
print("All checks passed")

### Exercise: Information and the counting window

Write `joint_table(T)` for 8 equally likely orientations and a neuron with f(θ) = 2 + 30 exp(2(cos 2(θ − 90°) − 1)) spikes per second, using Poisson probabilities for counts 0 to 80 (computed with `lgamma` in logs). For windows 0.05, 0.1, 0.25, 0.5 and 1 s, compute the information `info` and the bits per spike `bits_per_spike`.

In [ ]:
thetas = np.arange(8) * 22.5
f = 2 + 30 * np.exp(2 * (np.cos(np.deg2rad(2 * (thetas - 90))) - 1))
kk = np.arange(81)
log_fact = np.array([lgamma(k + 1) for k in kk])

def joint_table(T):
    lam = f * T
    logp = kk[None, :] * np.log(lam[:, None]) - lam[:, None] - log_fact[None, :]
    P = np.exp(logp)
    return ...                                     # each row is P(count | theta) / 8

windows = [0.05, 0.1, 0.25, 0.5, 1.0]
info = [mutual_info(joint_table(T)) for T in windows]
bits_per_spike = ...
print("information (bits):", np.round(info, 3)); print("bits per spike:   ", np.round(bits_per_spike, 3))

In [ ]:
import numpy as np
def test_joint():
    "joint_table returns a normalised 8 x K table"
    J = joint_table(0.5)
    assert J.shape[0] == 8 and abs(J.sum() - 1) < 1e-9

def test_sublinear():
    "information rises with the window but less than doubles from 0.25 to 0.5 s, and never exceeds 3 bits"
    i1, i2 = info[windows.index(0.25)], info[windows.index(0.5)]
    assert i1 < i2 < 2 * i1 and max(info) <= 3

def test_bps():
    "bits per spike fall as the window grows"
    assert bits_per_spike[0] > bits_per_spike[-1]

test_joint()
test_sublinear()
test_bps()
print("All checks passed")

### Exercise: Decoding loses information

Two neurons with preferred orientations 45° and 135° respond to 8 orientations (window 0.25 s, counts 0 to 30). Build the joint table of orientation and the pair of counts (flatten the pair into one response index), compute `info_counts`, the information of each neuron alone (`info_n1`, `info_n2`), and the information carried by a maximum-likelihood decoded label (`info_decoded`).

In [ ]:
kk2 = np.arange(31); lf2 = np.array([lgamma(k + 1) for k in kk2])
def pois_rows(rates, T=0.25):
    lam = rates * T
    P = np.exp(kk2[None] * np.log(lam[:, None]) - lam[:, None] - lf2[None])
    return P / P.sum(axis=1, keepdims=True)
f1 = 2 + 30 * np.exp(2 * (np.cos(np.deg2rad(2 * (thetas - 45))) - 1))
f2 = 2 + 30 * np.exp(2 * (np.cos(np.deg2rad(2 * (thetas - 135))) - 1))
P1, P2 = pois_rows(f1), pois_rows(f2)                         # 8 x 31 each
joint_pair = (P1[:, :, None] * P2[:, None, :]).reshape(8, -1) / 8     # independent given theta
info_counts = mutual_info(joint_pair)
info_n1, info_n2 = mutual_info(P1 / 8), mutual_info(P2 / 8)

decoded = ...                                       # for each response pair, the theta index with the largest likelihood
joint_dec = np.zeros((8, 8))
for d in range(8):
    joint_dec[:, d] = joint_pair[:, decoded == d].sum(axis=1)
info_decoded = mutual_info(joint_dec)
print(f"neuron 1 alone {info_n1:.3f}, neuron 2 alone {info_n2:.3f}, both {info_counts:.3f}, decoded label {info_decoded:.3f} bits")

In [ ]:
import numpy as np
def test_dpi():
    "the decoded label carries less information than the two counts, and at least some"
    assert 0 < info_decoded < info_counts

def test_both_neurons():
    "two neurons together carry more information than either alone"
    assert info_counts > max(info_n1, info_n2)

test_dpi()
test_both_neurons()
print("All checks passed")